# MedPilot - Qwen2.5 on Colab

**Runtime -> GPU**

In [ ]:
!pip install fastapi uvicorn transformers pyngrok -q

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from fastapi import FastAPI
from pyngrok import ngrok
import uvicorn

NGROK_TOKEN = "3BJuRXeiwgSNG5zWfm89ESHTDtg_3QBZ3CU3f7doQ3a6stpQS"
ngrok.set_auth_token(NGROK_TOKEN)

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True)
print("Done!")

In [ ]:
import threading
import uvicorn

app = FastAPI()

def generate(prompt, max_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_tokens)
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

@app.post("/v1/chat/completions")
def chat(data: dict):
    messages = data.get("messages", [])
    prompt = messages[-1].get("content", "") if messages else ""
    return {"choices": [{"message": {"role": "assistant", "content": generate(prompt)}}]}

# Start ngrok
public_url = ngrok.connect(8000)
print(f"URL: {public_url}/v1/chat/completions")

# Chay server trong thread
def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    server.run()

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("Server is running!")
print("Giu tab nay mo!")